# Track record
Every preview ends with a probability that can be checked. After the event, fill in `outcome` (1 or 0) and rerun:
the Brier score compares our calls with the market's. Skill above 0 means we beat the market.

In [2]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from altinfo import gap, satellite, earnings, scoring, charts, valuation, eventbeta as eb
from altinfo.charts import NAVY, BLUE, RED, GREY
charts.style()
RAW = ROOT / "data" / "raw"
SNAP = json.load(open(ROOT / "data" / "snapshot_2026-11-09.json"))
LIVE = (RAW / "prices.csv").exists()
print("Mode:", "LIVE (data/raw)" if LIVE else "SNAPSHOT (figures of 9 Nov 2026; run scripts/fetch_data.py for live data)")
def kalshi_live(series):
    """Open Kalshi markets of a series with a mid price (bid/ask midpoint, else last trade)."""
    km = pd.read_csv(RAW / "kalshi_markets.csv")
    m = km[km["series"] == series].copy()
    mid = (m["yes_bid_dollars"] + m["yes_ask_dollars"]) / 2
    tight = (m["yes_ask_dollars"] - m["yes_bid_dollars"]).le(0.10) & m["yes_ask_dollars"].gt(0)
    m["mid"] = mid.where(tight, m["last_price_dollars"])
    return m
def poly_live(pattern):
    pm = pd.read_csv(RAW / "polymarket_markets.csv")
    return pm[pm.question.str.contains(pattern, case=False, na=False, regex=True) & (pm.closed != True)].sort_values("volume", ascending=False)
def live_calls(today="2026-11-10"):
    """The calls registered in the reports, with the market probability updated to the latest live data."""
    calls = pd.DataFrame(SNAP["calls"]).assign(asset=["NVDA", "BRENT"], date=["2026-11-17", "2026-11-30"],
                                                confidence="medium", date_asof=pd.Timestamp(today))
    calls["market_week_ago"] = [SNAP["nvda"]["real"]["polymarket_beat_history"]["2026-11-03"], np.nan]
    if LIVE:
        cd = pd.read_csv(RAW / "kalshi_candles.csv", parse_dates=["date"])
        cd = cd[cd.ticker.str.startswith("KXBRENTMON-26NOV30")].copy()
        cd["strike"] = cd.ticker.str.extract(r"-T([\d.]+)$")[0].astype(float)
        daily = cd.dropna(subset=["price"]).groupby("date").apply(lambda d: gap.ladder_prob_above(d.strike, d.price, 96), include_groups=False)
        bm = kalshi_live("KXBRENTMON").dropna(subset=["floor_strike", "mid"])
        calls.loc[calls.asset == "BRENT", "market"] = gap.ladder_prob_above(bm.floor_strike, bm.mid, 96)
        calls.loc[calls.asset == "BRENT", "market_week_ago"] = daily.loc[:pd.Timestamp(today) - pd.Timedelta(days=7)].iloc[-1]
        nb_ = poly_live(r"N[Vv][Ii][Dd][Ii][Aa].*beat|NVDA.*beat")
        if len(nb_):
            calls.loc[calls.asset == "NVDA", "market"] = float(nb_["price"].iloc[0])
    return calls
def load_prices():
    p = pd.read_csv(RAW / "prices.csv")
    p["date"] = pd.to_datetime(p["date"], format="mixed").dt.normalize()
    return p.pivot_table(index="date", columns="ticker", values="close").sort_index()

Mode: LIVE (data/raw)


In [3]:
calls = pd.DataFrame(SNAP["calls"])
# Fill in after the events, e.g.:
# calls.loc[calls.event.str.startswith("NVDA"), "outcome"] = 1   # EPS above consensus on 17 Nov
# calls.loc[calls.event.str.startswith("Brent"), "outcome"] = 0  # Kalshi settlement on 30 Nov
scored, summary = scoring.track_record(calls)
# Brier scores exist only once an event has resolved: until then the row shows "pending"
view = pd.DataFrame({
    "Event": scored["event"],
    "Our probability": scored["ours"].map("{:.0%}".format),
    "Market probability": scored["market"].map("{:.0%}".format),
    "Market source": scored["source_market"],
    "Outcome": scored["outcome"].map(lambda o: "pending" if o is None or o != o else ("yes" if o == 1 else "no")),
    "Brier, ours": scored.get("brier_ours", pd.Series(index=scored.index, dtype=float)).map(lambda x: "pending" if x != x else f"{x:.3f}"),
    "Brier, market": scored.get("brier_market", pd.Series(index=scored.index, dtype=float)).map(lambda x: "pending" if x != x else f"{x:.3f}"),
})
display(view)
print(f"Resolved calls: {summary['resolved']} of {len(calls)}. Brier scores appear after NVIDIA reports (17 Nov) and Kalshi settles Brent (30 Nov).")

Resolved calls: 0 of 2. Brier scores appear after NVIDIA reports (17 Nov) and Kalshi settles Brent (30 Nov).


Event,Our probability,Market probability,Market source,Outcome,"Brier, ours","Brier, market"
NVDA Q3 FY27 EPS above consensus (17 Nov 2026),90%,92%,Polymarket,pending,pending,pending
Brent above $95.99 on 30 Nov 2026,62%,52%,Kalshi,pending,pending,pending


**How to read it.** The two calls made in the reports of 9 November: our probability, the market's and where the market price comes from. The Brier score measures how good a probability was once the outcome is known: 0 is perfect, 0.25 is a coin toss, so lower is better. It stays 'pending' until the event happens.

In [4]:
# Until then: what each possible outcome would do to the score (the two calls of 9 Nov 2026)
rows = []
for o1 in (0, 1):
    for o2 in (0, 1):
        c = calls.copy(); c["outcome"] = [o1, o2]
        _, s = scoring.track_record(c)
        rows.append({"NVIDIA beats": "yes" if o1 else "no", "Brent above $96": "yes" if o2 else "no",
                     "Brier, ours": round(s["brier_ours"], 3), "Brier, market": round(s["brier_market"], 3),
                     "Skill vs market": round(s["skill_vs_market"], 2)})
pd.DataFrame(rows)

NVIDIA beats,Brent above $96,"Brier, ours","Brier, market",Skill vs market
no,no,0.597,0.558,-0.07
no,yes,0.477,0.538,0.11
yes,no,0.197,0.138,-0.42
yes,yes,0.077,0.118,0.35


**How to read it.** What the scores would be for each of the four possible outcomes. Skill compares us with the market: above 0 means our probabilities were closer to what happened than the market's, below 0 that the market was better.